# Espectrometría gamma: calibración en energía y borde Compton

Espectros de rayos gamma tomados con un detector de centelleo y un analizador
multicanal, que los guarda en formato `.spe`.

**Qué se hace acá**

1. Se parsea el `.spe` (`leer_spe`): el bloque de cuentas por canal vive entre
   los delimitadores `$DATA:` y `$ROI:`.
2. Se calibra canal → energía con una fuente de Cs-137, cuyo fotopico está en
   661,7 keV. La pendiente `m` y la ordenada `b` vienen de ese ajuste, con sus
   incertezas.
3. Se resta el fondo ambiente medido por separado, truncando en cero.
4. Se ajusta el **borde Compton** con una gaussiana convolucionada con un escalón
   (`scipy.special.erf`): esa es la forma que toma un corte abrupto en energía
   cuando lo mira un detector de resolución finita.
5. Se repite para absorbentes de plomo, cobre y madera, y se superponen los
   espectros para comparar la atenuación.

Los errores en las cuentas son de Poisson (`σ_N = √N`), y los ajustes usan
`absolute_sigma=True` para que las incertezas de los parámetros salgan en
unidades físicas y no reescaladas.

*Universidad Nacional de La Plata · Licenciatura en Física*


In [ ]:
(from google.colab import drive
drive.mount('/content/drive')

# Cs

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.special import erf

# ==========================
# === CONFIGURACIÓN ===
# ==========================

m = 0.94575167
sigma_m = 0.00022661
b = -5.57280133
sigma_b = 0.13342535

# ==========================
# === FUNCIONES ===
# ==========================

def leer_spe(ruta):
    """Lee un archivo .spe (Canal vs Cuentas)"""
    with open(ruta, 'r') as f:
        lines = f.readlines()
    start = lines.index('$DATA:\n') + 2
    end = lines.index('$ROI:\n')
    counts = np.array([int(x.strip()) for x in lines[start:end]])
    canales = np.arange(len(counts))
    return canales, counts


def propagar_error_energia(canal, m, sigma_m, b, sigma_b):
    """Propagación de error en energía"""
    sigma_E = np.sqrt((canal * sigma_m) ** 2 + sigma_b ** 2)
    return sigma_E


def restar_fondo(cuentas, fondo):
    """Resta de fondo (evitando negativos)"""
    cuentas_corr = cuentas - fondo
    cuentas_corr[cuentas_corr < 0] = 0
    return cuentas_corr


def gauss_convol_step(E, A, E0, sigma, offset):
    """Convolución de una gaussiana con un escalón decreciente"""
    return offset + A * 0.5 * (1 - erf((E - E0) / (np.sqrt(2)*sigma)))


# ==========================
# === PROCESAMIENTO ===
# ==========================

ruta_muestra = "/content/Cs137-2.Spe"
ruta_fondo   = "/content/Fondo-17-9-4700.Spe"

# Leer espectros
canales_muestra, cuentas_muestra = leer_spe(ruta_muestra)
_, cuentas_fondo = leer_spe(ruta_fondo)

# Cortar desde canal 16 a 900
canales_muestra = canales_muestra[16:900]
cuentas_muestra = cuentas_muestra[16:900]
cuentas_fondo   = cuentas_fondo[16:900]

# Resta de fondo
cuentas_corr = restar_fondo(cuentas_muestra, cuentas_fondo)

# Calibración en Energía
E = m * canales_muestra + b
sigma_E = propagar_error_energia(canales_muestra, m, sigma_m, b, sigma_b)

# Error estadístico
sigma_N = np.sqrt(cuentas_corr)
sigma_N[sigma_N == 0] = 1

# ==========================
# === RANGO DE AJUSTE ===
# ==========================

ini, fin = 400, 560  # canales de ajuste
x_fit = E[ini:fin]
y_fit = cuentas_corr[ini:fin]
yerr_fit = sigma_N[ini:fin]

print(f"Usando {len(x_fit)} puntos desde el canal {ini} al {fin-1}")

# ==========================
# === AJUSTE ===
# ==========================

A0 = np.max(y_fit) - np.min(y_fit)
E0_0 = x_fit[np.argmin(y_fit)]
sigma0 = (x_fit[-1] - x_fit[0]) / 10
offset0 = np.min(y_fit)

p0 = [A0, E0_0, sigma0, offset0]

bounds = (
    [0, x_fit[0], 0, 0],
    [3*np.max(y_fit), x_fit[-1], (x_fit[-1] - x_fit[0]), np.max(y_fit)*1.2]
)

popt, pcov = curve_fit(
    gauss_convol_step,
    x_fit, y_fit,
    p0=p0,
    sigma=yerr_fit,
    absolute_sigma=True,
    bounds=bounds,
    maxfev=20000
)

perr = np.sqrt(np.diag(pcov))
A_fit, E0_fit, sigma_fit, offset_fit = popt



print(f"A      = {A_fit:.2f} ± {perr[0]:.2f}")
print(f"E0     = {E0_fit:.2f} ± {perr[1]:.2f}")
print(f"sigma  = {sigma_fit:.2f} ± {perr[2]:.2f}")
print(f"offset = {offset_fit:.2f} ± {perr[3]:.2f}")

# ==========================
# === GRÁFICO ===
# ==========================

plt.figure(figsize=(9,6), dpi=600)
plt.errorbar(x_fit, y_fit, yerr=yerr_fit, fmt='o', ms=4,
             color='tab:blue', ecolor='gray', elinewidth=1, capsize=2,
             label='Datos [372-538] keV')

plt.plot(x_fit, gauss_convol_step(x_fit, *popt), 'r-', lw=2, label='Ajuste')

plt.xlabel("Energía [keV]")
plt.ylabel("Cuentas")
plt.title(f"Ajuste - Borde Compton")
plt.legend()
plt.grid(True, which='both', linestyle='--', linewidth=1, alpha=1)
plt.tight_layout()

# Mostrar E0 dentro del gráfico
plt.text(0.94 * max(x_fit), 0.92 * max(y_fit),
         f"$E_c$ = {E0_fit:.2f} ± {perr[1]+0.23:.2f} keV",
         color='black', fontsize=11,
         bbox=dict(facecolor='white', alpha=0.7, edgecolor='red'))

plt.show()


# errorx

In [ ]:
# Calibración en Energía
E = (m+sigma_m) * canales_muestra + (b + sigma_b)
sigma_E = propagar_error_energia(canales_muestra, m, sigma_m, b, sigma_b)

# Error estadístico
sigma_N = np.sqrt(cuentas_corr)
sigma_N[sigma_N == 0] = 1

# ==========================
# === RANGO DE AJUSTE ===
# ==========================

ini, fin = 400, 560  # canales de ajuste
x_fit = E[ini:fin]
y_fit = cuentas_corr[ini:fin]
yerr_fit = sigma_N[ini:fin]

print(f"Usando {len(x_fit)} puntos desde el canal {ini} al {fin-1}")

# ==========================
# === AJUSTE ===
# ==========================

A0 = np.max(y_fit) - np.min(y_fit)
E0_0 = x_fit[np.argmin(y_fit)]
sigma0 = (x_fit[-1] - x_fit[0]) / 10
offset0 = np.min(y_fit)

p0 = [A0, E0_0, sigma0, offset0]

bounds = (
    [0, x_fit[0], 0, 0],
    [3*np.max(y_fit), x_fit[-1], (x_fit[-1] - x_fit[0]), np.max(y_fit)*1.2]
)

popt, pcov = curve_fit(
    gauss_convol_step,
    x_fit, y_fit,
    p0=p0,
    sigma=yerr_fit,
    absolute_sigma=True,
    bounds=bounds,
    maxfev=20000
)

perr = np.sqrt(np.diag(pcov))
A_fit, E0_fit, sigma_fit, offset_fit = popt

print(f"A      = {A_fit:.2f} ± {perr[0]:.2f}")
print(f"E0     = {E0_fit:.2f} ± {perr[1]:.2f}")
print(f"sigma  = {sigma_fit:.2f} ± {perr[2]:.2f}")
print(f"offset = {offset_fit:.2f} ± {perr[3]:.2f}")

# ==========================
# === GRÁFICO ===
# ==========================

plt.figure(figsize=(10,6))
plt.errorbar(x_fit, y_fit, yerr=yerr_fit, fmt='o', ms=4,
             color='tab:purple', ecolor='gray', elinewidth=1, capsize=2,
             label='Datos (rango de ajuste)')

plt.plot(x_fit, gauss_convol_step(x_fit, *popt), 'r-', lw=2, label='Ajuste')

plt.xlabel("Energía [keV]")
plt.ylabel("Cuentas")
plt.title(f"Ajuste Gaussiana * Escalón (canales {ini}–{fin})")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
sigma_Ec= 476.48 - 475.99 + 0.44
print(sigma_Ec)

In [ ]:
# Calibración en Energía
E = (m - sigma_m) * canales_muestra + (b - sigma_b)
sigma_E = propagar_error_energia(canales_muestra, m, sigma_m, b, sigma_b)

# Error estadístico
sigma_N = np.sqrt(cuentas_corr)
sigma_N[sigma_N == 0] = 1

# ==========================
# === RANGO DE AJUSTE ===
# ==========================

ini, fin = 400, 560  # canales de ajuste
x_fit = E[ini:fin]
y_fit = cuentas_corr[ini:fin]
yerr_fit = sigma_N[ini:fin]

print(f"Usando {len(x_fit)} puntos desde el canal {ini} al {fin-1}")

# ==========================
# === AJUSTE ===
# ==========================

A0 = np.max(y_fit) - np.min(y_fit)
E0_0 = x_fit[np.argmin(y_fit)]
sigma0 = (x_fit[-1] - x_fit[0]) / 10
offset0 = np.min(y_fit)

p0 = [A0, E0_0, sigma0, offset0]

bounds = (
    [0, x_fit[0], 0, 0],
    [3*np.max(y_fit), x_fit[-1], (x_fit[-1] - x_fit[0]), np.max(y_fit)*1.2]
)

popt, pcov = curve_fit(
    gauss_convol_step,
    x_fit, y_fit,
    p0=p0,
    sigma=yerr_fit,
    absolute_sigma=True,
    bounds=bounds,
    maxfev=20000
)

perr = np.sqrt(np.diag(pcov))
A_fit, E0_fit, sigma_fit, offset_fit = popt

print(f"A      = {A_fit:.2f} ± {perr[0]:.2f}")
print(f"E0     = {E0_fit:.2f} ± {perr[1]:.2f}")
print(f"sigma  = {sigma_fit:.2f} ± {perr[2]:.2f}")
print(f"offset = {offset_fit:.2f} ± {perr[3]:.2f}")

# ==========================
# === GRÁFICO ===
# ==========================

plt.figure(figsize=(10,6))
plt.errorbar(x_fit, y_fit, yerr=yerr_fit, fmt='o', ms=4,
             color='tab:purple', ecolor='gray', elinewidth=1, capsize=2,
             label='Datos (rango de ajuste)')

plt.plot(x_fit, gauss_convol_step(x_fit, *popt), 'r-', lw=2, label='Ajuste')

plt.xlabel("Energía [keV]")
plt.ylabel("Cuentas")
plt.title(f"Ajuste Borde Compton (canales {ini}–{fin})")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


# Pb


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ==========================
# === CONFIGURACIÓN ===
# ==========================

# ⚠️ COMPLETAR ESTOS VALORES CUANDO LOS TENGAS:
m = 0.968661     # pendiente (keV/canal por ej.)
sigma_m = 0.00016552 # error de la pendiente
b = -5.88706971        # ordenada al origen
sigma_b = 0.08230481  # error de la ordenada

# ==========================
# === FUNCIONES ===
# ==========================

def leer_spe(ruta):
    """Lee un archivo .spe (Canal vs Cuentas)"""
    with open(ruta, 'r') as f:
        lines = f.readlines()

    start = lines.index('$DATA:\n') + 2
    end = lines.index('$ROI:\n')
    counts = np.array([int(x.strip()) for x in lines[start:end]])
    canales = np.arange(len(counts))
    return canales, counts


def propagar_error_energia(canal, m, sigma_m, b, sigma_b):
    """Propagación de error en energía"""
    sigma_E = np.sqrt((canal * sigma_m) ** 2 + sigma_b ** 2)
    return sigma_E


def restar_fondo(cuentas, fondo):
    """Resta de fondo (evitando negativos)"""
    cuentas_corr = cuentas - fondo
    cuentas_corr[cuentas_corr < 0] = 0
    return cuentas_corr


def graficar(E, N, sigma_E, sigma_N, titulo="Espectro calibrado"):
    """Gráfico del espectro calibrado"""
    plt.figure(figsize=(10,6))
    plt.errorbar(E, N, xerr=sigma_E, yerr=sigma_N, fmt='o', ms=3,
                 color='tab:purple', ecolor='gray', elinewidth=1, capsize=2,
                 label="Datos calibrados")
    plt.xlabel("Energía [keV]")
    plt.ylabel("Cuentas [N]")
    plt.title(titulo)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


# ==========================
# === PROCESAMIENTO ===
# ==========================

# ⚠️ MODIFICAR RUTAS UNA VEZ SUBAS LOS ARCHIVOS
ruta_muestra = "/content/Cs137-Pb.Spe"
ruta_fondo   = "/content/Fondo-17-9-4700.Spe"

# Leer ambos espectros
canales_muestra, cuentas_muestra = leer_spe(ruta_muestra)
_, cuentas_fondo = leer_spe(ruta_fondo)

# 🔹 Cortar desde canal 16 en adelante
canales_muestra = canales_muestra[60:120]
cuentas_muestra = cuentas_muestra[60:120]
cuentas_fondo   = cuentas_fondo[60:120]

# Resta de fondo
cuentas_corr = restar_fondo(cuentas_muestra, cuentas_fondo)

# Calibración en Energía
E = m * canales_muestra + b
sigma_E = propagar_error_energia(canales_muestra, m, sigma_m, b, sigma_b)

# Error estadístico en las cuentas
sigma_N = np.sqrt(cuentas_corr)

from scipy.optimize import curve_fit
import numpy as np
import matplotlib.pyplot as plt

from scipy.optimize import curve_fit
import numpy as np
import matplotlib.pyplot as plt

# Modelo: Gaussiana convolucionada con una recta
def modelo(x, A, mu, sigma, m, b):
    return A * np.exp(-(x - mu)**2 / (2 * sigma**2)) + m * x + b

# Ajuste del modelo a los datos
def ajustar_espectro(E, N, sigma_E, sigma_N):
    # Ajustar el modelo a los datos
    popt, pcov = curve_fit(modelo, E, N, sigma=sigma_N, p0=[max(N), E[np.argmax(N)], 1, 0, 0])

    # Desempaquetar los parámetros ajustados
    A, mu, sigma, m, b = popt

    # Calcular los errores de los parámetros (diagonal de la matriz de covarianza)
    sigma_A, sigma_mu, sigma_sigma, sigma_m, sigma_b = np.sqrt(np.diag(pcov))

    # Calcular los valores ajustados
    N_ajustada = modelo(E, *popt)

    # Graficar
    plt.figure(figsize=(10, 6))
    plt.errorbar(E, N, yerr=sigma_N, fmt='o', ms=3, color='tab:purple', ecolor='gray', elinewidth=1, capsize=2, label="Datos")
    plt.plot(E, N_ajustada, 'r-', label="Ajuste: Gaussiana + Recta", linewidth=2)
    plt.xlabel("Energía [keV]")
    plt.ylabel("Cuentas [N]")
    plt.title("Ajuste Gaussiana + Recta")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    # Imprimir los resultados del ajuste
    print("\n--- Resultados del ajuste ---")
    print(f"A (Amplitud del pico): {A:.3f} ± {sigma_A:.3f}")
    print(f"μ (Posición del pico): {mu:.3f} ± {sigma_mu:.3f} keV")
    print(f"σ (Ancho del pico): {sigma:.3f} ± {sigma_sigma:.3f} keV")
    print(f"m (Pendiente de la recta): {m:.3f} ± {sigma_m:.3f}")
    print(f"b (Ordenada al origen): {b:.3f} ± {sigma_b:.3f}")

    return popt, pcov



# ==========================
# === PROCESAMIENTO ===
# ==========================

# Llamada a la función de ajuste
ajustar_espectro(E, cuentas_corr, sigma_E, sigma_N)

# ==========================
# === GRÁFICO FINAL ===
# ==========================




# Cu

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ==========================
# === CONFIGURACIÓN ===
# ==========================

# ⚠️ COMPLETAR ESTOS VALORES CUANDO LOS TENGAS:
m = 0.99999539     # pendiente (keV/canal por ej.)
sigma_m = 0.00026316 # error de la pendiente
b = -5.84192476        # ordenada al origen
sigma_b = 0.14327433  # error de la ordenada

# ==========================
# === FUNCIONES ===
# ==========================

def leer_spe(ruta):
    """Lee un archivo .spe (Canal vs Cuentas)"""
    with open(ruta, 'r') as f:
        lines = f.readlines()

    start = lines.index('$DATA:\n') + 2
    end = lines.index('$ROI:\n')
    counts = np.array([int(x.strip()) for x in lines[start:end]])
    canales = np.arange(len(counts))
    return canales, counts


def propagar_error_energia(canal, m, sigma_m, b, sigma_b):
    """Propagación de error en energía"""
    sigma_E = np.sqrt((canal * sigma_m) ** 2 + sigma_b ** 2)
    return sigma_E


def restar_fondo(cuentas, fondo):
    """Resta de fondo (evitando negativos)"""
    cuentas_corr = cuentas - fondo
    cuentas_corr[cuentas_corr < 0] = 0
    return cuentas_corr


def graficar(E, N, sigma_E, sigma_N, titulo="Espectro calibrado"):
    """Gráfico del espectro calibrado"""
    plt.figure(figsize=(10,6))
    plt.errorbar(E, N, xerr=sigma_E, yerr=sigma_N, fmt='o', ms=3,
                 color='tab:purple', ecolor='gray', elinewidth=1, capsize=2,
                 label="Datos calibrados")
    plt.xlabel("Energía [keV]")
    plt.ylabel("Cuentas [N]")
    plt.title(titulo)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


# ==========================
# === PROCESAMIENTO ===
# ==========================

# ⚠️ MODIFICAR RUTAS UNA VEZ SUBAS LOS ARCHIVOS
ruta_muestra = "/content/Cs137-cu.Spe"
ruta_fondo   = "/content/Fondo-17-9-4700.Spe"

# Leer ambos espectros
canales_muestra, cuentas_muestra = leer_spe(ruta_muestra)
_, cuentas_fondo = leer_spe(ruta_fondo)

# 🔹 Cortar desde canal 16 en adelante
canales_muestra = canales_muestra[16:900]
cuentas_muestra = cuentas_muestra[16:900]
cuentas_fondo   = cuentas_fondo[16:900]

# Resta de fondo
cuentas_corr = restar_fondo(cuentas_muestra, cuentas_fondo)

# Calibración en Energía
E = m * canales_muestra + b
sigma_E = propagar_error_energia(canales_muestra, m, sigma_m, b, sigma_b)

# Error estadístico en las cuentas
sigma_N = np.sqrt(cuentas_corr)

# ==========================
# === GRÁFICO FINAL ===
# ==========================
graficar(E, cuentas_corr, sigma_E, sigma_N, titulo="Espectro calibrado (fondo restado)")




# Madera

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ==========================
# === CONFIGURACIÓN ===
# ==========================

# ⚠️ COMPLETAR ESTOS VALORES CUANDO LOS TENGAS:
m = 0.9466762     # pendiente (keV/canal por ej.)
sigma_m = 0.00027283 # error de la pendiente
b = -5.27838409        # ordenada al origen
sigma_b = 0.13662442  # error de la ordenada

# ==========================
# === FUNCIONES ===
# ==========================

def leer_spe(ruta):
    """Lee un archivo .spe (Canal vs Cuentas)"""
    with open(ruta, 'r') as f:
        lines = f.readlines()

    start = lines.index('$DATA:\n') + 2
    end = lines.index('$ROI:\n')
    counts = np.array([int(x.strip()) for x in lines[start:end]])
    canales = np.arange(len(counts))
    return canales, counts


def propagar_error_energia(canal, m, sigma_m, b, sigma_b):
    """Propagación de error en energía"""
    sigma_E = np.sqrt((canal * sigma_m) ** 2 + sigma_b ** 2)
    return sigma_E


def restar_fondo(cuentas, fondo):
    """Resta de fondo (evitando negativos)"""
    cuentas_corr = cuentas - fondo
    cuentas_corr[cuentas_corr < 0] = 0
    return cuentas_corr


def graficar(E, N, sigma_E, sigma_N, titulo="Espectro calibrado"):
    """Gráfico del espectro calibrado"""
    plt.figure(figsize=(10,6))
    plt.errorbar(E, N, xerr=sigma_E, yerr=sigma_N, fmt='o', ms=3,
                 color='tab:purple', ecolor='gray', elinewidth=1, capsize=2,
                 label="Datos calibrados")
    plt.xlabel("Energía [keV]")
    plt.ylabel("Cuentas [N]")
    plt.title(titulo)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


# ==========================
# === PROCESAMIENTO ===
# ==========================

# ⚠️ MODIFICAR RUTAS UNA VEZ SUBAS LOS ARCHIVOS
ruta_muestra = "/content/Cs137-madera.Spe"
ruta_fondo   = "/content/Fondo-17-9-4700.Spe"

# Leer ambos espectros
canales_muestra, cuentas_muestra = leer_spe(ruta_muestra)
_, cuentas_fondo = leer_spe(ruta_fondo)

# 🔹 Cortar desde canal 16 en adelante
canales_muestra = canales_muestra[16:900]
cuentas_muestra = cuentas_muestra[16:900]
cuentas_fondo   = cuentas_fondo[16:900]

# Resta de fondo
cuentas_corr = restar_fondo(cuentas_muestra, cuentas_fondo)

# Calibración en Energía
E = m * canales_muestra + b
sigma_E = propagar_error_energia(canales_muestra, m, sigma_m, b, sigma_b)

# Error estadístico en las cuentas
sigma_N = np.sqrt(cuentas_corr)

# ==========================
# === GRÁFICO FINAL ===
# ==========================
graficar(E, cuentas_corr, sigma_E, sigma_N, titulo="Espectro calibrado (fondo restado)")


# superposición de todos los espectros

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ==========================
# === DATOS DE CALIBRACIÓN ===
# ==========================
# Cada espectro tiene su propia pendiente (m) y ordenada (b)
# y sus respectivas incertidumbres

calibraciones = {
    "Cs":     {"m": 0.94575167, "sigma_m": 0.00022661, "b": -5.57280133, "sigma_b": 0.13342535},
    "Pb":     {"m": 0.968661, "sigma_m": 0.00016552, "b": -5.60124562, "sigma_b": 0.08230481},
    "Cu":     {"m": 0.99999539, "sigma_m": 0.00026316, "b": -5.84192476, "sigma_b": 0.14327433},
    "Madera": {"m": 0.9466762, "sigma_m": 0.00027283, "b": -5.27838409, "sigma_b": 0.13662442}
}

# ==========================
# === RUTAS DE ARCHIVOS ===
# ==========================
# ⚠️ Cambiar las rutas a las de tu Google Drive o tu carpeta local

rutas = {
    "Cs":     "/content/Cs137-2.Spe",
    "Pb":     "/content/Cs137-Pb.Spe",
    "Cu":     "/content/Cs137-cu.Spe",
    "Madera": "/content/Cs137-madera.Spe",
    "Fondo":  "/content/Fondo-17-9-4700.Spe"
}

# ==========================
# === FUNCIONES ===
# ==========================

def leer_spe(ruta):
    """Lee un archivo .spe y devuelve arrays de canal y cuentas"""
    with open(ruta, 'r') as f:
        lines = f.readlines()
    start = lines.index('$DATA:\n') + 2
    end = lines.index('$ROI:\n')
    counts = np.array([int(x.strip()) for x in lines[start:end]])
    canales = np.arange(len(counts))
    return canales, counts

def propagar_error_energia(canal, m, sigma_m, b, sigma_b):
    """Propagación de incertidumbre en la energía"""
    return np.sqrt((canal * sigma_m)**2 + sigma_b**2)

def restar_fondo(cuentas, fondo):
    """Resta de fondo (evitando valores negativos)"""
    corr = cuentas - fondo
    corr[corr < 0] = 0
    return corr

def graficar_espectros(espectros):
    """Grafica varios espectros calibrados con barras de error"""
    plt.figure(figsize=(10,6))

    colores = {
        "Cs": "tab:blue",
        "Pb": "tab:red",
        "Cu": "tab:green",
        "Madera": "tab:purple"
    }

    for nombre, datos in espectros.items():
        E, N, sigma_E, sigma_N = datos
        plt.errorbar(E, N, xerr=sigma_E, yerr=sigma_N,
                     fmt='o', ms=3, capsize=2, elinewidth=1,
                     label=nombre, color=colores.get(nombre, "gray"))

    plt.xlabel("Energía [keV]")
    plt.ylabel("Cuentas [N]")
    plt.title("Espectros calibrados con fondo restado")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# ==========================
# === PROCESAMIENTO ===
# ==========================

# Leer fondo
canales_fondo, cuentas_fondo = leer_spe(rutas["Fondo"])

# Diccionario para guardar los espectros calibrados
espectros = {}

for nombre in ["Cs", "Pb", "Cu", "Madera"]:
    canales, cuentas = leer_spe(rutas[nombre])

    # Cortar desde canal 16
    canales = canales[16:900]
    cuentas = cuentas[16:900]
    fondo = cuentas_fondo[16:900]

    # Resta de fondo
    cuentas_corr = restar_fondo(cuentas, fondo)

    # Calibración de energía
    cal = calibraciones[nombre]
    m, sm, b, sb = cal["m"], cal["sigma_m"], cal["b"], cal["sigma_b"]
    E = m * canales + b
    sigma_E = propagar_error_energia(canales, m, sm, b, sb)

    # Error estadístico
    sigma_N = np.sqrt(cuentas_corr)

    # Guardar resultados
    espectros[nombre] = (E, cuentas_corr, sigma_E, sigma_N)

# ==========================
# === GRÁFICO FINAL ===
# ==========================
graficar_espectros(espectros)


# Borde Compton

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.special import erf

canales_muestra = canales_muestra[400:600]
E = m * canales_muestra + b
sigma_E = propagar_error_energia(canales_muestra, m, sigma_m, b, sigma_b)

# Error estadístico en las cuentas
sigma_N = np.sqrt(cuentas_corr)

import numpy as np
from scipy.optimize import curve_fit
from scipy.special import erf
import matplotlib.pyplot as plt

# ---- función analítica ----
def gauss_convol_step(x, A, x0, sigma):
    return A * 0.5 * (1 + erf((x - x0) / (np.sqrt(2)*sigma)))

# ---- datos de ejemplo ----
x = E
A_real, x0_real, sigma_real = 3, 0.5, 0.8
y_err = sigma_N
y_data = cuentas_corr

# ---- definimos intervalo ----
x_min, x_max = 400, 600   # por ejemplo, solo ajustar entre -1 y 3
mask = (x >= x_min) & (x <= x_max)

x_fit = x[mask]
y_fit = y_data[mask]
yerr_fit = y_err[mask]

# ---- ajuste ----
p0 = [2.5, 0.0, 1.0]
popt, pcov = curve_fit(
    gauss_convol_step,
    x_fit,
    y_fit,
    p0=p0,
    sigma=yerr_fit,
    absolute_sigma=True
)

# ---- resultados ----
perr = np.sqrt(np.diag(pcov))
print(f"A = {popt[0]:.3f} ± {perr[0]:.3f}")
print(f"x0 = {popt[1]:.3f} ± {perr[1]:.3f}")
print(f"sigma = {popt[2]:.3f} ± {perr[2]:.3f}")

# ---- gráfico ----
plt.errorbar(x, y_data, yerr=y_err, fmt='o', alpha=0.4, label='Datos totales')
plt.errorbar(x_fit, y_fit, yerr=yerr_fit, fmt='o', label='Datos usados')
plt.plot(x, gauss_convol_step(x, *popt), 'r-', label='Ajuste (solo en intervalo)')
plt.axvline(x_min, color='k', ls='--', lw=0.8)
plt.axvline(x_max, color='k', ls='--', lw=0.8)
plt.legend()
plt.show()



# a

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ==========================
# === CONFIGURACIÓN ===
# ==========================

# ⚠️ COMPLETAR ESTOS VALORES CUANDO LOS TENGAS:
m = 0.99999539     # pendiente (keV/canal por ej.)
sigma_m = 0.00026316 # error de la pendiente
b = -5.84192476        # ordenada al origen
sigma_b = 0.14327433  # error de la ordenada

# ==========================
# === FUNCIONES ===
# ==========================

def leer_spe(ruta):
    """Lee un archivo .spe (Canal vs Cuentas)"""
    with open(ruta, 'r') as f:
        lines = f.readlines()

    start = lines.index('$DATA:\n') + 2
    end = lines.index('$ROI:\n')
    counts = np.array([int(x.strip()) for x in lines[start:end]])
    canales = np.arange(len(counts))
    return canales, counts


def propagar_error_energia(canal, m, sigma_m, b, sigma_b):
    """Propagación de error en energía"""
    sigma_E = np.sqrt((canal * sigma_m) ** 2 + sigma_b ** 2)
    return sigma_E


def restar_fondo(cuentas, fondo):
    """Resta de fondo (evitando negativos)"""
    cuentas_corr = cuentas - fondo
    cuentas_corr[cuentas_corr < 0] = 0
    return cuentas_corr


def graficar(E, N, sigma_E, sigma_N, titulo="Espectro calibrado"):
    """Gráfico del espectro calibrado"""
    plt.figure(figsize=(10,6))
    plt.errorbar(E, N, xerr=sigma_E, yerr=sigma_N, fmt='o', ms=3,
                 color='tab:purple', ecolor='gray', elinewidth=1, capsize=2,
                 label="Datos calibrados")
    plt.xlabel("Energía [keV]")
    plt.ylabel("Cuentas [N]")
    plt.title(titulo)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


# ==========================
# === PROCESAMIENTO ===
# ==========================

# ⚠️ MODIFICAR RUTAS UNA VEZ SUBAS LOS ARCHIVOS
ruta_muestra = "/content/Cs137-cu.Spe"
ruta_fondo   = "/content/Fondo-17-9-4700.Spe"

# Leer ambos espectros
canales_muestra, cuentas_muestra = leer_spe(ruta_muestra)
_, cuentas_fondo = leer_spe(ruta_fondo)

# 🔹 Cortar desde canal 16 en adelante
canales_muestra = canales_muestra[16:900]
cuentas_muestra = cuentas_muestra[16:900]
cuentas_fondo   = cuentas_fondo[16:900]

# Resta de fondo
cuentas_corr = restar_fondo(cuentas_muestra, cuentas_fondo)
N = cuentas_corr = restar_fondo(cuentas_muestra, cuentas_fondo)
# Calibración en Energía
E = (m + sigma_m)* canales_muestra + (b + sigma_b)
sigma_E = propagar_error_energia(canales_muestra, m, sigma_m, b, sigma_b)

# Error estadístico en las cuentas
sigma_N = np.sqrt(cuentas_corr)

# ==========================
# === GRÁFICO FINAL ===
# ==========================
graficar(E, cuentas_corr, sigma_E, sigma_N, titulo="Espectro calibrado (fondo restado)")




In [ ]:
# ===================================================
# === AJUSTE DEL PICO CON FONDO CUADRÁTICO (OPCIÓN 1) ===
# ===================================================
from scipy.optimize import curve_fit

# Asumo que las variables E y N del espectro completo ya están cargadas
# a partir de tu código de procesamiento anterior.

def gauss_con_cuadratica(E, A, mu, sigma, c_lin, m_lin, b_lin):
    """
    Función de ajuste con un pico Gaussiano y un fondo cuadrático.
    """
    # Término del pico Gaussiano
    gauss = A * np.exp(-0.5 * ((E - mu) / sigma) ** 2)

    # Término del fondo cuadrático
    fondo = c_lin * E**2 + m_lin * E + b_lin

    return gauss + fondo

# --- 1. Seleccionar región del espectro ---
E_min, E_max = 120, 290
mask = (E >= E_min) & (E <= E_max)
E_fit = E[mask]
N_fit = N[mask]

# --- 2. Estimaciones iniciales ---
A0 = np.max(N_fit) - np.min(N_fit)
mu0 = E_fit[np.argmax(N_fit)]
sigma0 = 10      # Ancho estimado (keV)
c_lin0 = 0       # Empezamos con curvatura cero (como una recta)
m_lin0 = 0
b_lin0 = np.min(N_fit)
p0 = [A0, mu0, sigma0, c_lin0, m_lin0, b_lin0]

# --- 3. Errores estadísticos ---
sigma_N = np.sqrt(N_fit)
sigma_N[sigma_N == 0] = 1 # Evitar división por cero

# --- 4. Ajuste ponderado ---
popt, pcov = curve_fit(
    gauss_con_cuadratica,
    E_fit, N_fit,
    sigma=sigma_N,
    absolute_sigma=True,
    p0=p0,
    maxfev=10000
)

A, mu, sigma, c_lin, m_lin, b_lin = popt
sigma_popt = np.sqrt(np.diag(pcov))

# --- 5. Chi-cuadrado ---
N_pred = gauss_con_cuadratica(E_fit, *popt)
residuos = (N_fit - N_pred)
chi2 = np.sum((residuos / sigma_N) ** 2)
ndof = len(N_fit) - len(popt)
chi2_reducido = chi2 / ndof

# ➤ Coeficiente de determinación R²
ss_res = np.sum(residuos ** 2)
ss_tot = np.sum((N_fit - np.mean(N_fit)) ** 2)
R2 = 1 - (ss_res / ss_tot)

# --- 6. Resultados ---
print("=== Ajuste del pico con Fondo Cuadrático ===")
print(f"Amplitud A        = {A:.2f} ± {sigma_popt[0]:.2f}")
print(f"Media μ           = {mu:.2f} ± {sigma_popt[1]:.2f} keV")
print(f"Ancho σ           = {sigma:.2f} ± {sigma_popt[2]:.2f} keV")
print(f"Coef. cuadrático c= {c_lin:.2e} ± {sigma_popt[3]:.2e}") # <-- NUEVO PARÁMETRO
print(f"Pendiente fondo m = {m_lin:.2e} ± {sigma_popt[4]:.2e}")
print(f"Ordenada fondo b  = {b_lin:.2f} ± {sigma_popt[5]:.2f}")
print("-" * 35)
print(f"χ² = {chi2:.2f}")
print(f"χ² reducido = {chi2_reducido:.3f}")
print(f"R2 = {R2:.4f}")
print(f"Grados de libertad = {ndof}")

# --- 7. Curva ajustada ---
E_modelo = np.linspace(E_min, E_max, 1000)
N_modelo = gauss_con_cuadratica(E_modelo, *popt)

# --- 8. Gráfico ---
plt.figure(figsize=(9, 6), dpi=400)
plt.errorbar(E_fit, N_fit, yerr=sigma_N, fmt='o', ms=3,
             color='tab:green', ecolor='green', elinewidth=1, capsize=2,
             label="Datos experimentales")

plt.plot(E_modelo, N_modelo, 'r-', lw=2, label="Ajuste gauss + cuadrática")
plt.axvline(mu, color='tab:blue', ls='--', lw=1,
            label=f"μ = {mu:.1f} ± {sigma_popt[1]:.1f} keV")

plt.xlabel("Energía [keV]")
plt.ylabel("Cuentas [N]")
plt.title("Pico de retrodispersión")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()